# Tool-diversity ablation: does KD substitute for tool diversity?

Needs GPU T4 x2, Internet, and the `HF_TOKEN` secret ticked. Research question: the main
pipeline always trains on 8 tools; does the KD advantage over plain SFT shrink or grow as the
training data covers FEWER distinct tools (2, 4), holding the total example count (800) fixed?
See `adbench.evaluation.ablation_worker`'s module docstring for the full design.

`n_tools=8` is **not retrained** here — it is exactly the main pipeline's existing sft_only/distilled
results (already on Hugging Face for seeds 0-4), since `--n-tools 8` with the same per-tool cap
and split seed reproduces the same data. This notebook only needs to produce `n_tools` in `{2, 4}`.

Two GPU workers train + evaluate their share of the (seed, n_tools, condition) jobs. Each job is
a full training run (not just eval), so this is more expensive per job than notebooks 11/13 — it
**stops starting new jobs after `MAX_MINUTES`** (per worker) and resumes on a later run (finished
jobs are skipped, checked against Hugging Face).

In [ ]:
SEEDS = [0, 1, 2]                      # secondary ablation: fewer seeds than the main 5, adjust if time allows
CONDITIONS = ["distilled", "sft_only"]  # the two ends of the KD-vs-plain-SFT question
N_TOOLS_VALUES = [2, 4]                 # 8 already exists (main pipeline) -- not rerun here
MAX_MINUTES = 150                       # stop starting new jobs after this long (per worker)

In [ ]:
import base64
import os
import subprocess
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Kaggle secrets (Add-ons -> Secrets), all optional:
#   GH_TOKEN  read access to the GitHub repo, needed only while the repo is private
#   HF_TOKEN  write access to a Hugging Face repo, needed only to resume across sessions
os.environ.setdefault('ADBENCH_HF_REPO', 'NahlaNabil/adbench-run')
os.environ.setdefault('ADBENCH_RUN_TAG', 'v1-fixed')
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ('GH_TOKEN', 'HF_TOKEN'):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
        except Exception:
            pass
except Exception:
    pass


def git(*args, timeout=600):
    """Run git without ever prompting (a credentials prompt would hang an unattended run for
    hours). Uses GH_TOKEN when set; if that fails (revoked token, or a public repo that needs
    none) it retries once without it."""
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
    token = os.environ.get('GH_TOKEN')
    for use_token in ([True, False] if token else [False]):
        cmd = ['git']
        if use_token:
            basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            cmd += ['-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}']
        try:
            subprocess.run(cmd + list(args), check=True, timeout=timeout, env=env)
            return
        except subprocess.CalledProcessError:
            if not use_token:
                raise
            print('git with GH_TOKEN failed; retrying without it.')


def run_module(*args, timeout=4 * 3600):
    """Run `python -m <args>` in a fresh process, print the tail of its output, and raise if it
    fails or exceeds `timeout` seconds (a bare `!` command never stops the notebook)."""
    proc = subprocess.run(
        [sys.executable, '-m', *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=timeout
    )
    print(proc.stdout[-20000:])
    proc.check_returncode()


ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    git('clone', 'https://github.com/Nahla-Nabil/agentic-distillation-benchmark.git', REPO_DIR)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True, timeout=1800)

src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
# re-sync to the latest commit
git('-C', REPO_DIR, 'checkout', '--', '.')
git('-C', REPO_DIR, 'pull')

## 1. Build the 2-tool and 4-tool data splits (CPU only, no GPU needed)

In [ ]:
for n in N_TOOLS_VALUES:
    run_module("adbench.data.prepare", "--config", "configs/data.yaml", "--n-tools", str(n))

## 2. Check the token can write

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    raise SystemExit("HF_TOKEN is not set: tick it under Add-ons -> Secrets for this notebook, then run again.")
os.environ["ADBENCH_RUN_TAG"] = "v2-seed0"
from adbench import pipeline_state as ps
ps.check_upload()      # raises if the token cannot write

## 3. Train + evaluate, two GPU workers

In [ ]:
import subprocess
import sys
import threading

# condition-major, so both workers get a mix of the expensive (distilled, loads the 14B teacher)
# and cheap (sft_only) jobs -- see notebooks/11_ext_eval.ipynb's job-list comment for why a
# seed/n_tools-major order can silently give one worker only the slow condition.
jobs = [
    f"{seed}:{n}:{cond}"
    for cond in CONDITIONS
    for n in N_TOOLS_VALUES
    for seed in SEEDS
]
print(len(jobs), "jobs:", jobs)


def pump(proc, name):
    for line in proc.stdout:
        if "Loading weights" in line or "it/s]" in line:
            continue
        print(f"[{name}] {line.rstrip()}", flush=True)


workers = []
for gpu in (0, 1):
    share = ",".join(jobs[gpu::2])
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        [sys.executable, "-m", "adbench.evaluation.ablation_worker", "--jobs", share,
         "--work-dir", f"/kaggle/working/ablation_w{gpu}", "--max-minutes", str(MAX_MINUTES)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
    )
    thread = threading.Thread(target=pump, args=(proc, f"gpu{gpu}"), daemon=True)
    thread.start()
    workers.append((proc, thread))

for proc, thread in workers:
    proc.wait()
    thread.join(timeout=30)
codes = [proc.returncode for proc, _ in workers]
print("worker exit codes:", codes)
if any(codes):
    raise RuntimeError("a worker failed; see the [gpuN] lines above")

## Summary of what was evaluated

In [ ]:
import glob
import json

import pandas as pd

rows = []
for path in glob.glob("/kaggle/working/ablation_w*/results/*.json"):
    rows += json.load(open(path, encoding="utf-8"))
if rows:
    df = pd.DataFrame(rows)
    table = df.groupby(["condition", "n_tools", "seed", "chain_length"]).success.mean().unstack("chain_length").round(3)
    print(table)
else:
    print("no result files in this session (everything was already on Hugging Face)")